# Comprehensive analysis: Llama 3.1 8B

Single-run collection of all data needed for cross-domain transfer, scaling,
control sensitivity, and signal decomposition. Collects once, analyzes many times.

Resolves two limitations: frontier scale (8B) and cross-domain transfer.

Runtime > Change runtime type > A100 GPU.

In [ ]:
!git clone https://github.com/tmcarmichael/nn-observability.git
%cd nn-observability
!pip install uv -q
!uv sync --extra transformer -q

In [ ]:
import os
os.environ.pop('MPLBACKEND', None)

# HF auth for gated Llama model
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets')
except Exception:
    os.environ['HF_TOKEN'] = ''  # paste token here if not using secrets

# Write token to cache for subprocesses
token = os.environ.get('HF_TOKEN', '')
os.makedirs(os.path.expanduser('~/.cache/huggingface'), exist_ok=True)
with open(os.path.expanduser('~/.cache/huggingface/token'), 'w') as f:
    f.write(token)

In [ ]:
import sys
sys.path.insert(0, 'src')

import json
import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr, rankdata
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformer_observe import (
    collect_layer_data, load_wikitext, train_linear_binary,
    evaluate_head, bootstrap_ci, compute_hand_designed, _activation_entropy
)
from observe import partial_spearman, compute_loss_residuals

## Phase 1: Load model and data

In [ ]:
MODEL_ID = 'meta-llama/Llama-3.1-8B'

print(f'Loading {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2"
).cuda()
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
N_PARAMS = sum(p.numel() for p in model.parameters()) / 1e9
print(f'{N_PARAMS:.1f}B params, {N_LAYERS} layers, {HIDDEN_DIM} dim')

MAX_TRAIN = max(150 * HIDDEN_DIM, 200000)
MAX_TEST = MAX_TRAIN // 2
print(f'Token budget: {MAX_TRAIN} train, {MAX_TEST} test ({MAX_TRAIN/HIDDEN_DIM:.0f} ex/dim)')

# Sanity check
with torch.inference_mode():
    test_ids = tokenizer("test", return_tensors="pt")["input_ids"].cuda()
    test_out = model(test_ids, output_hidden_states=True)
    assert len(test_out.hidden_states) == N_LAYERS + 1
    print(f'Sanity check passed: {len(test_out.hidden_states)} hidden states')

In [ ]:
# Load all document sets
from datasets import load_dataset

print('Loading WikiText...')
wiki_train = load_wikitext('train', max_docs=2000)
wiki_test = load_wikitext('test', max_docs=500)
print(f'  WikiText: {len(wiki_train)} train, {len(wiki_test)} test')

print('Loading C4...')
c4_docs = []
ds = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
for i, row in enumerate(ds):
    if i < 50000: continue
    text = row['text'].strip()
    if len(text) > 100:
        c4_docs.append(text)
    if len(c4_docs) >= 500: break
print(f'  C4: {len(c4_docs)} test docs')

c4_train_docs = []
ds2 = load_dataset('allenai/c4', 'en', split='validation', streaming=True)
for row in ds2:
    text = row['text'].strip()
    if len(text) > 100:
        c4_train_docs.append(text)
    if len(c4_train_docs) >= 2000: break
print(f'  C4: {len(c4_train_docs)} train docs')

print('Loading CodeSearchNet...')
code_docs = []
ds3 = load_dataset('code_search_net', 'python', split='test', streaming=True, trust_remote_code=True)
for row in ds3:
    text = row['whole_func_string'].strip()
    if len(text) > 100:
        code_docs.append(text)
    if len(code_docs) >= 500: break
print(f'  Code: {len(code_docs)} test docs')

## Phase 2: Coarse layer sweep (find peak)

In [ ]:
# Coarse sweep: every 3rd layer, 1 seed
stride = max(1, N_LAYERS // 12)
sweep_layers = list(range(0, N_LAYERS, stride))
if N_LAYERS - 1 not in sweep_layers:
    sweep_layers.append(N_LAYERS - 1)

print(f'Coarse sweep: {len(sweep_layers)} layers (stride {stride})')
layer_profile = {}

for layer in sweep_layers:
    tr = collect_layer_data(model, tokenizer, wiki_train, layer, 'cuda', MAX_TRAIN)
    te = collect_layer_data(model, tokenizer, wiki_test, layer, 'cuda', MAX_TEST)
    head = train_linear_binary(tr, seed=42)
    _, rho, _ = evaluate_head(head, te)
    layer_profile[layer] = float(rho)
    print(f'  layer {layer:>2}: {rho:+.4f}')
    del tr, te
    torch.cuda.empty_cache()

coarse_peak = max(layer_profile, key=layer_profile.get)

# Dense sweep around peak
if stride > 1:
    dense_lo = max(0, coarse_peak - stride)
    dense_hi = min(N_LAYERS - 1, coarse_peak + stride)
    for layer in range(dense_lo, dense_hi + 1):
        if layer in layer_profile: continue
        tr = collect_layer_data(model, tokenizer, wiki_train, layer, 'cuda', MAX_TRAIN)
        te = collect_layer_data(model, tokenizer, wiki_test, layer, 'cuda', MAX_TEST)
        head = train_linear_binary(tr, seed=42)
        _, rho, _ = evaluate_head(head, te)
        layer_profile[layer] = float(rho)
        print(f'  layer {layer:>2}: {rho:+.4f}')
        del tr, te
        torch.cuda.empty_cache()

# Guard against peak at output
peak_layer = max(layer_profile, key=layer_profile.get)
output_layer = N_LAYERS - 1
if peak_layer >= output_layer - 1:
    max_mid = int(0.8 * N_LAYERS)
    mid_candidates = {l: r for l, r in layer_profile.items() if l <= max_mid}
    if mid_candidates:
        peak_layer = max(mid_candidates, key=mid_candidates.get)
        print(f'Global peak near output, using mid-depth peak: layer {peak_layer}')

print(f'\nPeak layer: {peak_layer} ({peak_layer/N_LAYERS:.0%} depth) = {layer_profile[peak_layer]:+.4f}')

## Phase 3: Collect all activations at peak + output layer (do once)

In [ ]:
print(f'Collecting activations at peak layer {peak_layer} and output layer {output_layer}...')
print('  WikiText train @ peak...')
wiki_train_peak = collect_layer_data(model, tokenizer, wiki_train, peak_layer, 'cuda', MAX_TRAIN)
print('  WikiText test @ peak...')
wiki_test_peak = collect_layer_data(model, tokenizer, wiki_test, peak_layer, 'cuda', MAX_TEST)
print('  WikiText test @ output...')
wiki_test_output = collect_layer_data(model, tokenizer, wiki_test, output_layer, 'cuda', MAX_TEST)
print('  WikiText train @ output...')
wiki_train_output = collect_layer_data(model, tokenizer, wiki_train, output_layer, 'cuda', MAX_TRAIN)
print('  C4 test @ peak...')
c4_test_peak = collect_layer_data(model, tokenizer, c4_docs, peak_layer, 'cuda', MAX_TEST)
print('  C4 train @ peak...')
c4_train_peak = collect_layer_data(model, tokenizer, c4_train_docs, peak_layer, 'cuda', MAX_TRAIN)
print('  Code test @ peak...')
code_test_peak = collect_layer_data(model, tokenizer, code_docs, peak_layer, 'cuda', MAX_TEST)
print('All activations collected.')

## Phase 4: Core analysis (all CPU from here, no more forward passes)

In [ ]:
# Free GPU memory
del model
import gc
gc.collect()
torch.cuda.empty_cache()
print('Model unloaded, GPU memory freed. All analysis runs on CPU from stored tensors.')

In [ ]:
# --- A: Three-seed battery at peak layer ---
print('=== A: Three-seed battery ===')
seeds = [42, 43, 44]
all_scores, all_rhos = [], []
heads = []

for seed in seeds:
    head = train_linear_binary(wiki_train_peak, seed=seed)
    scores, rho, p = evaluate_head(head, wiki_test_peak)
    all_scores.append(scores)
    all_rhos.append(float(rho))
    heads.append(head)
    print(f'  seed {seed}: partial corr = {rho:+.4f}')

pairwise = []
for i in range(len(seeds)):
    for j in range(i+1, len(seeds)):
        r, _ = spearmanr(all_scores[i], all_scores[j])
        pairwise.append(float(r))

mean_rho = np.mean(all_rhos)
mean_agree = np.mean(pairwise)
print(f'  Partial corr: {mean_rho:+.4f} +/- {np.std(all_rhos):.4f}')
print(f'  Seed agreement: {mean_agree:+.4f}')

In [ ]:
# --- B: Negative baselines ---
print('=== B: Negative baselines ===')
hand_designed = compute_hand_designed(wiki_test_peak)
for name, scores in hand_designed.items():
    rho_hd, _ = partial_spearman(
        scores, wiki_test_peak['losses'],
        [wiki_test_peak['max_softmax'], wiki_test_peak['activation_norm']]
    )
    print(f'  {name:<20} partial corr = {rho_hd:+.4f}')

# Random head
torch.manual_seed(99)
rand_head = torch.nn.Linear(HIDDEN_DIM, 1)
rand_head.eval()
with torch.inference_mode():
    rand_scores = rand_head(wiki_test_peak['activations']).squeeze(-1).numpy()
rho_rand, _ = partial_spearman(
    rand_scores, wiki_test_peak['losses'],
    [wiki_test_peak['max_softmax'], wiki_test_peak['activation_norm']]
)
print(f'  {"random_head":<20} partial corr = {rho_rand:+.4f}')

In [ ]:
# --- C: Output-controlled residual ---
print('=== C: Output-controlled residual ===')
ctrl_rhos = []
for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    n_feat = wiki_train_output['activations'].size(1)
    predictor = torch.nn.Sequential(
        torch.nn.Linear(n_feat, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1)
    )
    opt = torch.optim.Adam(predictor.parameters(), lr=1e-3, weight_decay=1e-4)
    ds = torch.utils.data.TensorDataset(
        wiki_train_output['activations'],
        torch.from_numpy(wiki_train_output['losses']).float()
    )
    dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
    for ep in range(20):
        for bx, by in dl:
            loss = F.mse_loss(predictor(bx).squeeze(-1), by)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
    predictor.eval()
    with torch.inference_mode():
        pred_scores = predictor(wiki_test_output['activations']).squeeze(-1).numpy()
    
    obs_head = train_linear_binary(wiki_train_peak, seed=seed)
    obs_head.eval()
    with torch.inference_mode():
        obs_scores = obs_head(wiki_test_peak['activations']).squeeze(-1).numpy()
    
    rho_ctrl, _ = partial_spearman(
        obs_scores, wiki_test_peak['losses'],
        [wiki_test_peak['max_softmax'], wiki_test_peak['activation_norm'], pred_scores]
    )
    ctrl_rhos.append(float(rho_ctrl))
    print(f'  seed {seed}: output-controlled = {rho_ctrl:+.4f}')

print(f'  Mean: {np.mean(ctrl_rhos):+.4f}')

In [ ]:
# --- D: Cross-domain transfer ---
print('=== D: Cross-domain transfer (WikiText-trained observer) ===')
domain_results = {}
for domain_name, test_data in [('wikitext', wiki_test_peak), ('c4', c4_test_peak), ('code', code_test_peak)]:
    rhos = []
    for head in heads:
        _, rho, _ = evaluate_head(head, test_data)
        rhos.append(float(rho))
    domain_results[domain_name] = float(np.mean(rhos))
    print(f'  {domain_name:<12}: {np.mean(rhos):+.4f}')

# Within-domain C4
print('\n  Within-domain C4 (train on C4, test on C4):')
c4_rhos = []
for seed in seeds:
    c4_head = train_linear_binary(c4_train_peak, seed=seed)
    _, rho, _ = evaluate_head(c4_head, c4_test_peak)
    c4_rhos.append(float(rho))
    print(f'    seed {seed}: {rho:+.4f}')
domain_results['c4_within'] = float(np.mean(c4_rhos))
print(f'  Within-domain C4: {np.mean(c4_rhos):+.4f}')

In [ ]:
# --- E: Control sensitivity ---
print('=== E: Control sensitivity ===')

# Nonlinear MLP control
torch.manual_seed(42)
conf_feats = torch.from_numpy(
    np.column_stack([wiki_train_peak['max_softmax'], wiki_train_peak['activation_norm']])
).float()
loss_tgt = torch.from_numpy(wiki_train_peak['losses']).float()
mlp_ctrl = torch.nn.Sequential(torch.nn.Linear(2, 64), torch.nn.ReLU(), torch.nn.Linear(64, 1))
opt = torch.optim.Adam(mlp_ctrl.parameters(), lr=1e-3, weight_decay=1e-4)
ds = torch.utils.data.TensorDataset(conf_feats, loss_tgt)
dl = torch.utils.data.DataLoader(ds, batch_size=1024, shuffle=True)
for ep in range(20):
    for bx, by in dl:
        loss = F.mse_loss(mlp_ctrl(bx).squeeze(-1), by)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
mlp_ctrl.eval()
test_conf = torch.from_numpy(
    np.column_stack([wiki_test_peak['max_softmax'], wiki_test_peak['activation_norm']])
).float()
with torch.inference_mode():
    mlp_pred = mlp_ctrl(test_conf).squeeze(-1).numpy()

# Use seed 42 observer
head = heads[0]
head.eval()
with torch.inference_mode():
    obs = head(wiki_test_peak['activations']).squeeze(-1).numpy()

td = wiki_test_peak
controls = {
    'none': [],
    'softmax_only': [td['max_softmax']],
    'norm_only': [td['activation_norm']],
    'standard': [td['max_softmax'], td['activation_norm']],
    'plus_entropy': [td['max_softmax'], td['activation_norm'], td['logit_entropy']],
    'nonlinear': [mlp_pred],
}

ctrl_results = {}
for name, covs in controls.items():
    if name == 'none':
        r, _ = spearmanr(obs, td['losses'])
    else:
        r, _ = partial_spearman(obs, td['losses'], covs)
    ctrl_results[name] = float(r)
    print(f'  {name:<16}: {r:+.4f}')

In [ ]:
# --- SUMMARY ---
print('\n' + '=' * 60)
print(f'  LLAMA 3.1 8B COMPREHENSIVE RESULTS')
print(f'  Peak layer: {peak_layer} ({peak_layer/N_LAYERS:.0%} depth)')
print('=' * 60)

print(f'\n  Partial corr:      {mean_rho:+.4f} +/- {np.std(all_rhos):.4f}')
print(f'  Seed agreement:    {mean_agree:+.4f}')
print(f'  Output-controlled: {np.mean(ctrl_rhos):+.4f}')

print(f'\n  Cross-domain transfer:')
for name, rho in domain_results.items():
    print(f'    {name:<15}: {rho:+.4f}')

print(f'\n  Control sensitivity:')
for name, rho in ctrl_results.items():
    print(f'    {name:<16}: {rho:+.4f}')

In [ ]:
# Save everything
output = {
    'model': MODEL_ID,
    'n_params_b': round(N_PARAMS, 1),
    'n_layers': N_LAYERS,
    'hidden_dim': HIDDEN_DIM,
    'peak_layer': peak_layer,
    'peak_layer_frac': round(peak_layer / N_LAYERS, 2),
    'layer_profile': {str(k): v for k, v in sorted(layer_profile.items())},
    'partial_corr': {'mean': float(mean_rho), 'per_seed': all_rhos},
    'seed_agreement': {'mean': float(mean_agree), 'pairwise': pairwise},
    'output_controlled': {'mean': float(np.mean(ctrl_rhos)), 'per_seed': ctrl_rhos},
    'cross_domain': domain_results,
    'control_sensitivity': ctrl_results,
}

with open('llama8b_results.json', 'w') as f:
    json.dump(output, f, indent=2)
print(json.dumps(output, indent=2))

from google.colab import files
files.download('llama8b_results.json')